# Module 4: Building Your Own Environment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/openenv-course/blob/main/module-4/notebook.ipynb)

In this notebook, you'll:
1. Scaffold a new environment with `openenv init`
2. Implement a word-guessing game
3. Test it locally
4. Deploy to HF Spaces

## Setup

In [ ]:
!pip install openenv-core fastapi uvicorn pydantic -q

## 1. Scaffold with `openenv init`

Create the basic structure for our word game environment.

In [ ]:
# Create a new environment
!openenv init word_game
%cd word_game
!tree -L 2 .

## 2. Define Types (models.py)

Let's define our action, observation, and state models.

In [ ]:
%%writefile word_game/models.py
from typing import List, Optional
from openenv.core.env_server import Action, Observation, State

class WordGameAction(Action):
    guess: str  # The player's guessed letter

class WordGameObservation(Observation):
    masked_word: str           # e.g., "h_ll_"
    guessed_letters: List[str] # Letters tried so far
    attempts_remaining: int
    message: str               # Feedback message

class WordGameState(State):
    target_word: str = ""
    max_attempts: int = 10

## 3. Implement Environment Logic (server/environment.py)

Now let's implement the game logic.

In [ ]:
%%writefile word_game/server/environment.py
import random
import uuid
from openenv.core.env_server import Environment
from ..models import WordGameAction, WordGameObservation, WordGameState

WORDS = ["python", "neural", "tensor", "matrix", "vector",
         "kernel", "lambda", "signal", "binary", "cipher"]

class WordGameEnvironment(Environment):
    SUPPORTS_CONCURRENT_SESSIONS = True
    MAX_ATTEMPTS = 10

    def __init__(self):
        self._state = WordGameState()
        self._target = ""
        self._guessed = set()
        self._remaining = self.MAX_ATTEMPTS

    def reset(self, seed=None, episode_id=None, **kwargs) -> WordGameObservation:
        self._target = random.choice(WORDS)
        self._guessed = set()
        self._remaining = self.MAX_ATTEMPTS
        self._state = WordGameState(
            episode_id=episode_id or str(uuid.uuid4()),
            step_count=0,
            target_word=self._target,
            max_attempts=self.MAX_ATTEMPTS,
        )
        return WordGameObservation(
            done=False,
            reward=None,
            masked_word=self._mask(),
            guessed_letters=[],
            attempts_remaining=self._remaining,
            message=f"Guess letters in a {len(self._target)}-letter word!",
        )

    def step(self, action: WordGameAction, timeout_s=None, **kwargs) -> WordGameObservation:
        letter = action.guess.lower().strip()
        self._state.step_count += 1
        self._guessed.add(letter)

        if letter in self._target:
            message = f"'{letter}' is in the word!"
        else:
            self._remaining -= 1
            message = f"'{letter}' is not in the word."

        masked = self._mask()
        won = "_" not in masked
        lost = self._remaining <= 0
        done = won or lost

        if won:
            reward = 1.0
            message = f"You got it! The word was '{self._target}'."
        elif lost:
            reward = 0.0
            message = f"Out of attempts. The word was '{self._target}'."
        else:
            reward = 0.0

        return WordGameObservation(
            done=done,
            reward=reward,
            masked_word=masked,
            guessed_letters=sorted(self._guessed),
            attempts_remaining=self._remaining,
            message=message,
        )

    @property
    def state(self) -> WordGameState:
        return self._state

    def _mask(self) -> str:
        return "".join(c if c in self._guessed else "_" for c in self._target)

## 4. Test Locally

Let's test our environment before deploying.

In [ ]:
# In a real terminal, you would run:
# cd word_game
# uv run server

# For this notebook, we'll simulate a game
from word_game.server.environment import WordGameEnvironment
from word_game.models import WordGameAction

# Create environment
env = WordGameEnvironment()

# Reset
obs = env.reset()
print(f"Starting game: {obs.message}")
print(f"Masked word: {obs.masked_word}")

# Play a few turns
for letter in ['e', 'a', 'o']:
    action = WordGameAction(guess=letter)
    obs = env.step(action)
    print(f"\nGuessed '{letter}': {obs.message}")
    print(f"Masked word: {obs.masked_word}")
    print(f"Attempts remaining: {obs.attempts_remaining}")
    
    if obs.done:
        print(f"\nGame over! Final reward: {obs.reward}")
        break

## 5. Deploy to HF Spaces

Once you're happy with your environment, deploy it:

```bash
# Login to HF
huggingface-cli login

# Deploy
cd word_game
openenv push --repo-id yourusername/word-game
```

Your environment will be available at:
- `https://yourusername-word-game.hf.space`

## Key Takeaways

1. **Standard pattern**: Types → Server → Client → Deploy
2. **Type safety**: Pydantic models catch bugs early
3. **Quick iteration**: Test locally before deploying
4. **One command deploy**: `openenv push` handles everything

You now know how to build custom RL environments! In Module 5, you'll learn how to train LLMs using these environments.

## Exercise: Extend the Game

Try adding:
1. Difficulty levels (easy/medium/hard word lists)
2. Hints after failed attempts
3. A scoring system based on attempts used
4. Multi-player mode

In [ ]:
# Your code here
